# Gentrex: extracting and standardizing clinical genetic test reports with local open-weight models

This notebook holds the text, tables and figures of the paper, in the order of the paper. Every number comes from the scored runs in `results/gentrex-bench/`. Each table is also written to `figures/` as CSV and LaTeX, and each figure as PDF and PNG. The sections after the discussion are the supplementary material.

In [ ]:
# pyright: reportMissingImports=false
import sys

from IPython.display import Markdown
from IPython.display import display

sys.path.insert(0, "evaluation_results")
import benchruns as bench  # noqa: E402

runs = bench.load_runs()
chosen, run = bench.start(runs=runs)

# The report that Supplement S2 follows through the pipeline, such as "SYN-0004".
REPORT = bench.example_report(run)
files, report_errors = bench.example(run, report=REPORT, runs=runs)

## Abstract

In [ ]:
bench.abstract(run, chosen)

## 1 Introduction

A genetic test report names the variants that were found, their ACMG classification and zygosity, the associated condition, and often the clinical features of the patient. Registries, research databases and electronic health records need these facts as structured data. Laboratories, however, send the reports as PDFs, and each laboratory uses its own layout. The finding may be on the second page of a long report, between the description of the assay, the list of genes on the panel and legal notices. Some reports arrive scanned or faxed.

Copying the results by hand is slow and introduces errors. Large language models can read such documents, but most hospitals are not allowed to send patient reports to a cloud service. Open-weight models that run on the hospital's own GPUs remove this obstacle. How accurate they are on long reports in many layouts, and how much accuracy scanned and faxed pages cost, has hardly been measured.

The closest earlier work is that of Yu, Weile and Courtot (2026). They generated 1,000 molecular test reports from seven LaTeX templates, each with a simulated fax copy, and benchmarked LLM-based extraction tools on them. The best local model reached an average F1 of 41.3, and the fax copies lowered F1 by 38.8%. The text of their reports was generated by code, and F1 was averaged only over the answers that were valid JSON, so a tool that often fails to answer is not penalized for it.

We present Gentrex, a pipeline for this task that runs entirely on local hardware with open-weight models (Figure 1). It splits the task into four steps, and each step is a command that can be run and checked on its own. Pages with a text layer skip OCR, and pages without findings never reach the extraction step. Every standard code is stored next to the wording of the report and the page it came from, and four automatic checks flag records for human review.

We also present gentrex-bench, a generator of synthetic reports with a scorer. The generator samples the findings from ClinVar, HGNC, MONDO and HPO before it renders the report, so the correct output of every step is known. The scorer can therefore charge each error to the step that caused it, and a report without an answer counts as wrong.

With both, we ask four questions.

1. How accurate is the pipeline on native PDFs?
2. How much accuracy do scans and faxes cost, and at which step is it lost?
3. What does page selection save, and what does it cost?
4. Do the automatic checks find the reports that are wrong?

![Figure 1](docs/images/gentrex-arch.svg)

**Figure 1.** The Gentrex pipeline. Each command reads the run folder of the command before it. `gentrex load` and `gentrex export` write the standardized records to a database or to exchange formats.

## 2 Methods

### 2.1 Pipeline

Gentrex runs four commands in turn, `gentrex transcribe`, `gentrex select`, `gentrex extract` and `gentrex normalize`. Each writes one JSONL file and a `run.yaml` with its settings, the git commit, its time and its token counts. The two GPU models are served one at a time, so `gentrex run` starts each vLLM server when needed, reuses the language model across `gentrex select` and `gentrex extract`, and stops it afterwards. With `--retry`, a command runs only the files that failed.

**Table 1.** The commands of Gentrex.

| Command | Reads | Writes | Model | Hardware |
| --- | --- | --- | --- | --- |
| `gentrex transcribe` | PDFs | `pages.jsonl` | LightOnOCR-3-4B, for pages without a usable text layer | 1 GPU |
| `gentrex select` | `pages.jsonl` | `decisions.jsonl` and the kept pages | Gemma 4 26B-A4B | 1 GPU |
| `gentrex extract` | Kept pages | `extractions.jsonl` | Gemma 4 26B-A4B | 1 GPU |
| `gentrex normalize` | `extractions.jsonl` | `normalized.jsonl` | SapBERT and EmbeddingGemma 2 | CPU |
| `gentrex load`, `gentrex export` | `normalized.jsonl` | SQL tables, Phenopackets, OMOP CDM, FHIR | None | CPU |

**Transcribe.** `gentrex transcribe` inspects each page with pdfplumber before it uses a GPU. A page goes to LightOnOCR-3-4B if its text layer has fewer than 20 characters once unmapped glyph codes such as `(cid:36)` are removed. It also goes to OCR if images cover at least half of it and it has fewer than 300 characters, or if it draws its text as vector outlines, which shows as at least 1,000 curves with fewer than 600 characters. All other pages are read from the text layer.

**Select.** `gentrex select` asks Gemma 4 26B-A4B three yes-or-no questions about each page. They ask whether the page states a test result, a clinical feature of the patient, or a date of the test. The model answers with one token, with its reasoning turned off. Gentrex adds up the probabilities of the spellings of "yes" and of "no" among the 20 most likely first tokens, and uses the share of "yes" as the page's score for that question. The page text comes before the question, so vLLM computes it once and reuses it for all three questions. A page is kept if any score reaches 0.5 (`select.keep_threshold`). A page whose request fails is kept, so that a failed request never removes a finding.

**Extract.** `gentrex extract` sends all kept pages of a report in one request to Gemma 4 26B-A4B, the model of `gentrex select`, at temperature 0. A request holds up to 32,768 tokens. The answer must follow a JSON schema (Table 2), which the server enforces while the model writes. Each field with a fixed list of values, such as the ACMG classification or the zygosity, comes right after a field for the laboratory's own wording. The model writes the fields in schema order, so it copies the wording before it picks the category. After extraction, Gentrex looks for each copied value in the page text, letters and digits only, and lists the values that it cannot find under `not_in_pages`. A value may run over a few lines, as a wrapped table cell does.

**Normalize.** `gentrex normalize` runs on the CPU. Rules turn dates into ISO 8601 and map the specimen and the overall result to fixed categories. Dates such as 06/12/2019 are read month first, as US laboratories write them, and `normalize.day_first` switches this. The overall result is compared with the classifications of the primary variants. Gene symbols become HGNC IDs, including previous symbols and aliases. HGVS expressions are checked with the Mutalyzer HGVS parser, and each variant is looked up in ClinVar. Conditions get a MONDO ID by an exact match on MONDO labels and exact synonyms. Clinical features are compared with every HPO name and synonym by two embedding models. SapBERT was trained on biomedical synonyms and EmbeddingGemma 2 on general text. Each model's cosine similarities are turned into z-scores and the two are added. The best term is assigned if both models reach a cosine similarity of 0.5, and the five best terms are kept for review.

**Table 2.** The fields of the extraction schema (`schema.yaml`). Every field may be empty, so the model can leave out what the report does not state.

In [ ]:
bench.schema_table()

### 2.2 Design rationale

**Separate steps instead of one request.** The simplest design sends the images of all pages to one vision-language model and asks for the record. We split the task because its parts have different needs. Reading a scanned page needs a vision model, but most pages already have a text layer. Deciding whether a page matters needs little reasoning, but runs on every page and has to be cheap. Extraction writes a long answer under a schema, but only for the pages with findings. Page selection and extraction use the same language model, so the pipeline needs two models, and each fits on one GPU with 80 GB. It therefore runs on a single hospital server, and the scorer can charge each error to one step. The price is that errors add up along the chain, because a value lost by an early step cannot be recovered later. Figure 5 measures this loss, and Table 12 tests the pipeline without page selection.

**Page routing instead of OCR for every page.** Where a text layer exists, it is exact. OCR can misread one character of an HGVS expression or a medical record number, and such an error is hard to notice. Gentrex therefore uses OCR only where the text layer is missing or unusable: scanned pages, pages that are one large image with a short line of text (six of the public sample reports carry only a document code as text), and pages whose text is drawn as outlines. A page is rendered at 400 dpi and reduced to 2,048 pixels on its longer side, matching LightOnOCR-3-4B training resolution while keeping small print smooth. The image is sent without a prompt, as in the model's training.

**Page scores from token probabilities.** A model that writes "Yes" or "No" gives only a hard decision. Reading the probability of its first token gives a calibrated score between 0 and 1 for the same single token. With a continuous score, the threshold can be moved after the run without calling the model again (Figure 6), and ranking quality can be measured directly (Table 10). We ask three narrow questions rather than one general question because pages are needed for different reasons: one states the result, another the patient's features, and a third the specimen or report dates. The date question was added because some reports place dates only in a footer or summary table on a separate page. The result question instructs the model to answer no for pages that only explain the result or recommend follow-up. Table 11 compares the three questions with one general question, and Table 13 shows what each question contributes.

**One extraction request per report.** Extracting page by page would require a merging step and cannot link a variant on one page to its classification on the next when a variant table splits across pages. Sending all kept pages together avoids both problems, while page selection keeps context length bounded. Guided JSON decoding guarantees that every completed response matches the schema. Answers truncated at the 4,096-token limit (enough for roughly 20 variants) fail the report cleanly rather than silently emitting partial records.

**The report's wording before the category.** Because vLLM writes fields in schema order, the model copies the laboratory's wording before choosing a category. This serves three purposes: the model commits to printed text before interpreting it, a reviewer sees the original wording beside every code, and `not_in_pages` can check the copied string against the page text. We did not ablate field order in isolation. Page numbers for variants are not requested from the model. Gentrex locates them deterministically by searching the pages for the DNA change.

**Rules and lookups where they suffice.** Dates, categories, gene symbols and conditions are normalized with deterministic rules and exact dictionary lookups. They are fast, reproducible and auditable to a single rule. Exact lookup leaves some condition names without a MONDO ID, but rarely assigns a wrong one (Table 19). For a clinical database, a missing code is safer than an incorrect diagnosis.

**Two embedding models for HPO.** Clinical features appear in free text and cannot rely on exact lookup. Because SapBERT (biomedical) and EmbeddingGemma 2 (general) produce cosine similarities on different scales, each model's scores are converted to z-scores before summing rather than fusing raw scores or ranks. Whether both models choose the same top term also provides a built-in review flag (`hpo_models_disagree`). We omit lexical BM25 fusion because HPO terms share generic words such as "abnormality" while missing morphological variants such as "ataxic" and "ataxia" (Table 18).

### 2.3 Benchmark reports

Real genetic reports contain patient data and cannot be published, and annotating hundreds of long PDFs by hand takes expert time. gentrex-bench generates synthetic reports instead, with a known truth for every step (Figure 2).

The generator samples the truth before it writes the report. Each case is drawn from public data. It draws a variant and its classification from ClinVar, the gene from HGNC, the condition from MONDO and the patient's features from the HPO annotations of the condition. It also plants distractors, such as a variant cited from the literature or a feature of a relative. A language model then writes the free text of the report from a short summary of the case. We used Muse Glimmer 30B for this, which belongs to another model family than Gemma, so that the extraction model does not read text from its own family. The case and the text fill one of 21 HTML templates, and Chrome prints it as a PDF.

The templates follow the layouts of public sample reports from seven laboratories, under invented names. They cover gene panels, exomes, genomes, single-gene and targeted-variant tests, carrier screens, the mitochondrial genome, microarrays, karyotypes, repeat expansions, SMN1 copy number and methylation tests. Each case also gets one or two of eight challenges. A challenge makes the report harder in one known way. Examples are a variant table split across two pages and IDs that look like the medical record number. Each challenge appears in about 50 of the 300 reports.

Real reports often come as packets with pages after the report itself. Each benchmark report therefore gets up to five appendix pages, on the methods and limitations of the test, information for patients, resources, billing and regulatory statements. These pages state none of the patient's findings.

Every PDF is checked against its truth before it is kept. Each value must be found on the page where the truth puts it. A report that fails is written again, with at most three attempts. Each kept report then gets two more copies, an office scan and a fax (Figure 3). The three copies share one truth and differ only in image quality.

![Figure 2](docs/images/benchmark-generator.svg)

**Figure 2.** How gentrex-bench makes a report. The truth is sampled first and the PDF is checked against it.

<img src="docs/images/benchmark-report-sources.png" width="430" alt="Page 1 of a synthetic report. Yellow marks values that code placed, red marks distractors, and blue marks text that the language model wrote.">

**Figure 3.** Page 1 of one report as a native PDF, an office scan and a fax. The scan has a slight skew, a little blur and noise, and JPEG compression. The fax has a lower vertical resolution, a larger skew, black and white pixels only, and speckles.

In [ ]:
bench.copies_figure()

**Table 3.** Composition of the benchmark. A stated value is a field that the report prints. Fields that a report leaves empty are not counted, so an empty answer earns nothing.

In [ ]:
bench.data_table(run)

Table 4 compares the length of the benchmark reports with that of the public sample reports that the templates follow.

**Table 4.** Length of the benchmark reports and of the public sample reports of the same laboratories. Words are counted only on reports in which every page has a text layer.

In [ ]:
bench.show_or_note(
    bench.realism_table(run),
    "The public sample reports are not in data/gentrex-bench/public/pdf.",
)

### 2.4 Evaluation

**Scoring.** The scorer compares each record with its truth, field by field. Text is compared after Unicode normalization, case folding, and removal of white space and trailing punctuation. Equivalent HGVS spellings count as equal, so c.5266dupC matches c.5266dup, and a reference sequence before the change is ignored. Table 14 also gives the F1 under exact matching. Variants are first matched by gene and DNA change, and their other attributes are scored afterwards. A report for which Gentrex wrote no record counts as wrong, and all its values count as missed.

The primary outcome is **report accuracy**, the share of reports in which every clinically relevant value is right. These values are the overall result, the karyotype, and the gene, DNA change, copy number, classification and zygosity of every variant. A report with an extra variant counts as wrong. We also report the share of reports in which every value is right, the share with the right MRN and date of birth, and three F1 scores.

- **Value F1** pools all stated values of all reports.
- **Field F1** is the mean of the F1 of each field, so that rare fields count as much as common ones.
- **Clinical variant F1** counts a variant as found only if its gene, DNA change, copy number, classification and zygosity are all right. **Variant detection F1** requires only the gene and the DNA change.
- **Pathogenic / likely pathogenic (P/LP) recall** measures locus detection and full clinical accuracy on actionable variants alone, and **actionable ACMG F1** evaluates the P/LP versus non-P/LP boundary separately from adjacent grade shifts such as Pathogenic versus Likely Pathogenic.

**Attribution of errors.** The scorer follows each stated value through the pipeline and charges a lost value to the first command that lost it. For page selection we report the areas under the ROC curve and the precision-recall curve of the page scores, with their Brier score and their expected calibration error over 10 bins.

**Statistics.** All intervals are 95% intervals. The main numbers use a bootstrap with 2,000 samples that resamples whole reports, so that the values of one report stay together. Two setups are compared on the same reports with a paired bootstrap and an exact McNemar test. Subgroups get Wilson intervals. Each challenge is tested with Fisher's exact test, and the eight p-values are adjusted with Holm's method. With 300 reports, a report accuracy of 90% has an interval of about ±3.4 points.

**Runs.** We made six runs on the 300 reports. The native PDFs were read three times, to measure how much identical runs differ. The scans and the faxes were read once each. One more run read the native PDFs without `gentrex select`, so that `gentrex extract` received every page. During the first native run, two other ways of choosing pages ran on the same pages and the same server. The first asks one general question per page. The second extracts a record from each page and keeps the pages whose record holds a finding.

**Development and test data.** Gentrex was developed on earlier reports from the same generator. The 300 reports evaluated here were sampled with a new seed after the settings of Gentrex had been fixed, and no setting was changed afterwards.

## 3 Results

### 3.1 Overall accuracy

The main numbers of the native PDFs are shown below, with the findings that follow from them. Table 5 gives them for every setup.

In [ ]:
bench.headline(run, chosen)

**Table 5.** Accuracy of each setup on the 300 reports, in percent, with 95% bootstrap intervals.

In [ ]:
bench.main_table(chosen)

Overall scores treat every error equally, but clinical risk is not uniform. Missing a pathogenic or likely pathogenic (P/LP) variant, flipping its classification across the actionable boundary, or reporting a variant on a negative test can alter care. Table 6 summarizes these safety measures alongside macro-F1 across result and ACMG classes.

**Table 6.** Clinical safety, P/LP sensitivity, grounding and class-balanced macro-F1 across setups.

In [ ]:
bench.safety_table(chosen)

Table 7 breaks down every clinical error by severity. Actionable ACMG flips cross the P/LP versus non-P/LP boundary. Other ACMG errors stay within the same clinical tier, such as Pathogenic versus Likely Pathogenic.

**Table 7.** Clinical errors by severity category across setups.

In [ ]:
bench.show_or_note(
    bench.clinical_errors_table(chosen),
    "Score the runs again with the current scorer to fill this table.",
)

### 3.2 Scans and faxes

The three copies of a report differ only in image quality, so a difference in accuracy between them is caused by reading the page (Figure 4). Figure 5 shows at which step the values are lost.

**Figure 4.** Report accuracy of each setup (left) and its paired difference to the native PDFs on the same reports (right), with 95% intervals.

In [ ]:
bench.accuracy_figure(chosen)

**Figure 5.** Stated values lost by `gentrex transcribe`, `gentrex select` and `gentrex extract`, per 100 stated values. Each value is charged to the first command that lost it.

In [ ]:
bench.stages_figure(chosen)

OCR reads the columns of a page in another order than the text layer does, so an edit distance would count correct text as wrong. Table 8 therefore compares the words of a page regardless of their order. It also counts the stated values that appear in the page text exactly, before extraction.

**Table 8.** Text and stated values that `gentrex transcribe` recovered from each copy.

In [ ]:
bench.transcribe_table(chosen)

In a real report, pages with a text layer, scanned pages and pages drawn as outlines can occur in one file. The synthetic copies are either all native or all scanned. Table 9 therefore applies the page routing of `gentrex transcribe` to the public sample reports, without a model.

**Table 9.** Routing of the pages of the public sample reports.

In [ ]:
bench.show_or_note(
    bench.public_reports_table(),
    "The public sample reports are not in data/gentrex-bench/public/pdf.",
)

### 3.3 Page selection

We call a page needed if it is the only page of its report that prints some stated value. If such a page is dropped, the value cannot be recovered.

**Table 10.** Pages kept and dropped by `gentrex select`, and how well its scores separate needed pages from the others.

In [ ]:
bench.select_pages_table(chosen)

**Table 11.** The three questions of `gentrex select` compared with one general question and with extracting a record from each page, on the same pages and the same server.

In [ ]:
bench.show_or_note(
    bench.select_methods_table(run),
    "The main run was made without --select-methods.",
)

**Table 12.** Accuracy and cost of the pipeline with and without `gentrex select`, on the same reports.

In [ ]:
bench.show_or_note(
    bench.select_table(chosen),
    "There is no run without gentrex select.",
)

Because the page scores are saved, other thresholds can be evaluated without running the model again (Figure 6).

**Figure 6.** Share of stated values on a kept page against the share of pages kept, for each threshold. The threshold of the run is marked.

In [ ]:
bench.select_figure(run)

**Table 13.** Pages kept by each question, and needed pages that only one question kept.

In [ ]:
bench.questions_table(run)

### 3.4 Extraction

To separate the extraction model from the commands before it, Table 14 scores only the values that reached `gentrex extract`, which are the values on a kept page that appear in the page text.

**Table 14.** Extraction of the values that reached `gentrex extract`, next to recall over all stated values.

In [ ]:
bench.extract_table(chosen)

**Table 15.** Variants and clinical features scored as whole entities. An extra entity is either a distractor that the report prints or one that appears nowhere in the report.

In [ ]:
bench.entities_table(run)

**Table 16.** Detection and attributes of variants, by kind of variant.

In [ ]:
bench.variant_kinds_table(run)

**Figure 7.** F1 of each field on the native PDFs, and on the faxes where that run exists, with the number of stated values. Exact precision, recall, F1 and exact-match rates for every field are given in Table S7.

In [ ]:
bench.fields_figure(run, chosen.get("fax"))

Figure 8 sorts the wrong values by kind. A wrong value can be missing, a part of the right value, another value of the same report, other text of the pages, or a wrong category or number.

**Figure 8.** The most frequent kinds of wrong values, by field.

In [ ]:
bench.errors_figure(run)

**Figure 9.** Confusions between the categories of the overall result and of the variant classification.

In [ ]:
bench.confusion_figure(run)

### 3.5 Normalization

Table 17 scores each standard code in two ways. The end-to-end score is what a user receives. The score given a right source leaves out the values that were already wrong before `gentrex normalize`, and so measures this command alone.

**Table 17.** Accuracy of the dates, categories and standard codes that `gentrex normalize` assigns.

In [ ]:
bench.normalize_table(run)

The synthetic reports name features and conditions with the wording of HPO and MONDO, which makes normalization easier than it is on real text. We therefore also tested both matchers on real wording. GSC+ holds 939 phenotype mentions from abstracts that experts mapped to HPO, and 626 of them are worded differently from any HPO name or synonym. On GSC+, z-score fusion of SapBERT and EmbeddingGemma 2 reached 67.1% top-1 accuracy (against 65.7% for an earlier rank-fusion version), and when the two models agreed on the top term (73.2% of mentions), 78.5% were right versus 36.1% when they disagreed. ClinVar names 9,278 distinct conditions across 2.85 million variants.

**Table 18.** HPO matching on GSC+, with exact McNemar tests against the default.

In [ ]:
bench.show_or_note(
    bench.hpo_benchmark_table(),
    "evaluation_results/hpo_normalization.json is not on this machine.",
)

**Table 19.** Exact MONDO lookup of the condition names in ClinVar.

In [ ]:
bench.show_or_note(
    bench.mondo_benchmark_table(),
    "evaluation_results/mondo_normalization.json is not on this machine.",
)

### 3.6 Subgroups

Figures 10 to 12 split report accuracy by the properties of the reports. Some subgroups hold only a few reports, so their intervals are wide.

**Figure 10.** Report accuracy by kind of test, result and number of pages, with 95% Wilson intervals and the number of reports.

In [ ]:
bench.groups_figure(run, ("kind", "scenario", "pages"))

**Figure 11.** Report accuracy by challenge, with 95% Wilson intervals and the number of reports.

In [ ]:
bench.groups_figure(run, ("challenge",), "figure-challenges")

**Table 20.** Report accuracy of the reports with and without each challenge, from the largest drop to the smallest, with Fisher's exact test.

In [ ]:
bench.show_or_note(
    bench.challenge_impact_table(run),
    "No challenge has reports both with and without it.",
)

**Figure 12.** Report accuracy by laboratory. Each laboratory has two to five templates, and Figure S1 shows them one by one.

In [ ]:
bench.groups_figure(run, ("laboratory",), "figure-laboratories")

### 3.7 Checks for human review

Gentrex flags a record when a copied value is not in the pages (`not_in_pages`), an HGVS expression is invalid (`hgvs_invalid`), the overall result disagrees with the variant classifications (`result_conflict`), or the two HPO models pick different terms (`hpo_models_disagree`).

**Table 21.** Precision and recall of each check, on the values that it examines.

In [ ]:
bench.checks_table(run)

**Table 22.** Review of flagged reports only. The table gives the reports flagged, the share of manual review avoided by accepting unflagged reports, the wrong reports caught, and the accuracy of the unflagged reports.

In [ ]:
bench.review_table(run)

### 3.8 Repeatability

A setup can give different records when it is run again, because batching on the GPU changes the order of floating-point operations.

**Table 23.** Main numbers over repeated runs with the same settings and code.

In [ ]:
bench.show_or_note(
    bench.repeats_table(runs),
    "No setup has been run more than once.",
)

### 3.9 Runtime, cost and outputs

**Table 24.** Completion rate, runtime, tokens and GPU-hours of each setup. Loading the models is counted once per run.

In [ ]:
bench.efficiency_table(chosen)

**Table 25.** The records of the main run loaded into SQLite and exported as GA4GH Phenopackets, with and without pseudonyms, and as HL7 FHIR Genomics Reporting bundles.

In [ ]:
bench.show_or_note(
    bench.export_summary_table(run),
    "The records of the main run are not on this machine.",
)

## 4 Discussion

The main findings, written from the scores:

In [ ]:
Markdown("\n".join(f"- {sentence}" for sentence in bench.findings(run, chosen)))

### What the design buys

Splitting the task into four commands has two practical benefits. Each error can be traced to the step that made it (Figure 5), and each step can be changed and tested without touching the others. The same split exposes where compute goes. Page selection spends one token per question and page, and Table 12 shows what it saves in the tokens that extraction has to read.

The pipeline is also built to fail visibly. A request that fails in `gentrex select` keeps the page so findings are never dropped silently. A truncated or malformed extraction fails the report cleanly, and the scorer counts every uncompleted report as wrong (Table 24). A copied value absent from the page text is listed in `not_in_pages`, invalid HGVS is flagged, result-classification conflicts are raised, and HPO model disagreements are recorded. Table 22 measures how much manual review is avoided by accepting unflagged reports and what accuracy remains on those unflagged reports. Together, these two numbers determine whether selective human review is clinically viable.

### Where errors come from

The attribution in Figure 5 separates three error sources that require different fixes. A value lost in `gentrex transcribe` is an OCR or scan-quality error. A value lost in `gentrex select` can be recovered by lowering `select.keep_threshold`, at the cost of sending more pages to extraction (Figure 6). A value lost in `gentrex extract` was visible to the model, and Figure 8 separates omissions, partial copies, and values taken from elsewhere on the page. The last pattern is typical of distractors such as literature variants.

Not all clinical errors carry the same patient-safety risk. Table 6 and Table 7 separate management-changing failures (missed P/LP variants, false-positive variants on negative tests, and actionable P/LP versus non-P/LP classification flips) from same-tier shifts such as Pathogenic versus Likely Pathogenic. Because the native PDF, scan and fax share identical content, their paired differences isolate image-quality loss (Figure 4), and Table 8 separates text dropped by OCR from text altered by OCR.

Compared with the closest prior benchmark (Yu, Weile and Courtot, 2026), two evaluation choices matter. First, scoring full clinical report accuracy and penalizing failed extractions prevents models with high schema-failure rates from appearing artificially strong. Second, evaluating native, scanned and faxed copies alongside real-text HPO and MONDO benchmarks (Tables 18 and 19) separates layout difficulty, image degradation and terminology normalization instead of collapsing them into a single F1 score.

### Limitations

The main limitation is that the reports are synthetic. They follow public sample reports, but they come from 21 templates that we wrote, and real laboratories vary more. Their appendix pages come from five fixed texts, whereas real appendices differ across laboratories (Table 4 compares lengths). Every stated value can be found in the text of the native PDF because the generator verifies this, which real reports do not guarantee. The scans and faxes are simulated from the native PDF, whereas clinical scans also carry handwriting, stamps and repeated photocopying.

The benchmark and the pipeline were built by the same authors, and Gentrex was developed on earlier reports from the same generator, so accuracy may be optimistic for unseen layouts. All cases are in English and follow US date conventions by default (`normalize.day_first` switches to day-first dates). The benchmark's features and conditions use HPO and MONDO wording, which makes normalization easier than on clinical notes (Table 18). The models come from two families only.

A study on real de-identified reports is needed before clinical deployment. Two annotators should label each report independently, with a clinical geneticist or genetic counselor resolving disagreements. Running additional open-weight model families on the same benchmark would test how much performance depends on Gemma 4 and LightOnOCR-3.

## Data and code availability

Gentrex, the benchmark generator, the scorer and this notebook are available at https://github.com/hieutdle/gentrex under the MIT license. The benchmark can be regenerated from the seed and reference files recorded in `manifest.json`. No patient data were used in this study. The benchmark reports are synthetic, and the public sample reports are published by the laboratories themselves.

## Supplementary material

### S1 Software and templates

**Table S1.** The software, models, hardware and reference files of the main run.

In [ ]:
bench.setup_table(run)

**Figure S1.** Report accuracy by template.

In [ ]:
bench.groups_figure(run, ("lab",), "figure-templates")

### S2 One report through the pipeline

**Table S2.** The report's template, test, result and challenges.

In [ ]:
bench.case_table(run, REPORT)

**Table S3.** How each page was read.

In [ ]:
bench.pages_table(files)

**Figure S2.** The pages with the score of each question. Kept pages have a blue frame, and dropped pages are faded.

In [ ]:
bench.pages_figure(files)

In [ ]:
bench.decisions_table(files)

**Figure S3.** Page 1 with the copied values boxed. Blue boxes are right and red boxes are wrong.

In [ ]:
bench.values_figure(files, report_errors, pages=1)

**Table S4.** The truth, the extracted record and the codes. Shaded rows are wrong.

In [ ]:
bench.record_table(files, report_errors)

In [ ]:
bench.show_or_note(
    bench.variants_table(files),
    "This report states no variants.",
)

In [ ]:
bench.show_or_note(
    bench.features_table(files),
    "This report states no clinical features.",
)

The report as a GA4GH Phenopacket, and as the resources of an HL7 FHIR bundle.

In [ ]:
bench.print_json(
    bench.phenopacket(run, files), "subject", "phenotypicFeatures", "interpretations"
)

In [ ]:
bench.fhir_table(files)

### S3 Speed of each command

**Table S5.** Speed of each command in each setup.

In [ ]:
for command in bench.COMMANDS:
    display(bench.speed_table(chosen, command))

### S4 HPO terms on the benchmark

**Table S6.** HPO terms for the clinical features whose text was extracted right.

In [ ]:
bench.hpo_table(run)

### S5 Queries on the loaded records

The first query counts patients with a pathogenic or likely pathogenic variant by gene. The second finds every feature under Seizure (HP:0001250) through the table of HPO ancestors. `gentrex export omop` writes each test to `PROCEDURE_OCCURRENCE`, each variant with its classification (LOINC 53037-8) and allelic state (LOINC 53034-5) to `MEASUREMENT`, and each feature to `OBSERVATION`.

In [ ]:
database = bench.database(run)
bench.query(
    database,
    """
    SELECT v.gene_symbol AS gene, count(DISTINCT r.patient_key) AS patients
    FROM variants v JOIN reports r ON r.file = v.file
    WHERE v.classification IN ('pathogenic', 'likely_pathogenic')
      AND v.gene_symbol IS NOT NULL
    GROUP BY v.gene_symbol ORDER BY patients DESC, gene LIMIT 5
    """,
)

In [ ]:
bench.query(
    database,
    """
    SELECT p.hpo_id, p.hpo_label AS feature, count(DISTINCT r.patient_key) AS patients
    FROM patient_phenotypes p
    JOIN hpo_ancestors a ON a.hpo_id = p.hpo_id
    JOIN reports r ON r.file = p.file
    WHERE a.ancestor_id = 'HP:0001250' AND p.present
    GROUP BY p.hpo_id, p.hpo_label ORDER BY patients DESC
    """,
)

### S6 Accuracy by field

**Table S7.** Precision, recall, F1, recall given input and exact-match rate for every schema field on the main run.

In [ ]:
bench.fields_table(run)

### S7 Every wrong report

**Table S8.** Each report of the main run that counts as wrong, with its clinical errors and the command that made each one.

In [ ]:
bench.show_or_note(
    bench.wrong_reports_table(run, bench.run_errors(run, runs)),
    "Every report of this run is right.",
)